In [4]:
%pip install python-frontmatter requests

Note: you may need to restart the kernel to use updated packages.


In [8]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(owner, name):
    # Download URL for the repository from GitHub as a ZIP file.
    url = f'https://codeload.github.com/{owner}/{name}/zip/refs/heads/master'
    print(f"Data is being downloaded: {owner}/{name}...")
    
    resp = requests.get(url)
    
    # If there is no master's branch, let's try the main branch.
    if resp.status_code != 200:
        url = url.replace('/master', '/main')
        resp = requests.get(url)
    
    if resp.status_code != 200:
        print("Error: Failed to download the repository. Please check your connection or the repository name.")
        return []

    repository_data = []
    # I unpack the downloaded content in memory like a ZIP file.
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        # I only target documentation files (.md or .mdx)
        if filename.lower().endswith(('.md', '.mdx')):
            try:
                with zf.open(file_info) as f_in:
                    content = f_in.read().decode('utf-8', errors='ignore')
                    # With `frontmatter.loads`, we separate both the metadata (if any) and the content.
                    post = frontmatter.loads(content)
                    data = post.to_dict()
                    data['filename'] = filename
                    data['content'] = post.content # Text content is stored here
                    repository_data.append(data)
            except Exception as e:
                # Silently skip corrupted files
                continue
    
    zf.close()
    return repository_data

# Homework test: Let's retrieve the FastAPI documentation.
docs = read_repo_data('tiangolo', 'fastapi')

print("-" * 30)
print(f"SUCCESS! A total of {len(docs)} documentation files have been processed.")

Data is being downloaded: tiangolo/fastapi...
------------------------------
SUCCESS! A total of 1554 documentation files have been processed.
